**Run evidence** · task `gold_sourcing` · run `765308892829766` · result **SUCCESS** · executed 2026-10-19 17:27 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `02_build_gold/02c_sourcing`.

# 02c — Workflow C → SOURCING (grain: MATNR + WERKS + LIFNR)

The heart of the sourcing-risk model: which vendors may supply which material at
which plant, plus their quota split.

**Steps (spec §4, Workflow C):**
1. Start from `RAW_EORD` (already at the target grain MATNR+WERKS+LIFNR).
2. Join `RAW_EQUK` to `RAW_EQUP` on EQUK_ID to flatten quota to MATNR+WERKS+LIFNR.
3. Left-join the flattened quota onto `RAW_EORD` using MATNR+WERKS+LIFNR.
4. **Allow QUOTA to be NULL** where a vendor is source-listed but not quota-split —
   a deliberate, meaningful gap (NOT a data error).
5. Output: one row per MATNR+WERKS+LIFNR with VDATU, BDATU, NOTKZ, FLIFN, QUOTA.

We also derive a convenience boolean `IS_VALID_SOURCE` (non-blocked source-list
entry) — this is what "valid, non-blocked vendor" means in the business questions.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
from pyspark.sql import functions as F

eord = spark.table(CFG.fq("RAW_EORD"))   # MATNR, WERKS, LIFNR, VDATU, BDATU, NOTKZ, FLIFN
equk = spark.table(CFG.fq("RAW_EQUK"))   # MATNR, WERKS, EQUK_ID
equp = spark.table(CFG.fq("RAW_EQUP"))   # EQUK_ID, LIFNR, QUOTA

# Step 2: flatten quota to MATNR+WERKS+LIFNR grain via the EQUK_ID bridge.
quota_flat = (
    equk.join(equp, on="EQUK_ID", how="inner")
        .select("MATNR", "WERKS", "LIFNR", "QUOTA")
)

# Step 3+4: left-join quota; QUOTA stays NULL for source-listed-but-not-quota vendors.
sourcing = (
    eord
    .join(quota_flat, on=["MATNR", "WERKS", "LIFNR"], how="left")
    .withColumn(
        # A source is "valid & non-blocked" when NOTKZ is not 'X'.
        "IS_VALID_SOURCE",
        F.when(F.coalesce(F.col("NOTKZ"), F.lit("")) != F.lit("X"), F.lit(True))
         .otherwise(F.lit(False)),
    )
    .select(
        "MATNR", "WERKS", "LIFNR", "VDATU", "BDATU", "NOTKZ", "FLIFN",
        "QUOTA", "IS_VALID_SOURCE",
    )
)

(sourcing.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(CFG.fq("SOURCING")))

cnt = spark.table(CFG.fq("SOURCING")).count()
dupes = (spark.table(CFG.fq("SOURCING"))
         .groupBy("MATNR", "WERKS", "LIFNR").count().filter("count > 1").count())
nullq = spark.table(CFG.fq("SOURCING")).filter("QUOTA IS NULL").count()
print(f"SOURCING rows: {cnt:,} | grain violations: {dupes} | NULL-quota (intended gaps): {nullq:,}")
assert dupes == 0, "SOURCING grain violated — MATNR+WERKS+LIFNR not unique!"
display(spark.table(CFG.fq("SOURCING")).limit(10))

SOURCING rows: 220 | grain violations: 0 | NULL-quota (intended gaps): 113


MATNR,WERKS,LIFNR,VDATU,BDATU,NOTKZ,FLIFN,QUOTA,IS_VALID_SOURCE
MAT100000,1010,V200012,2025-05-19,2027-01-01,,,None,True
MAT100000,1010,V200001,2025-12-07,2028-04-03,X,,None,False
MAT100002,1010,V200005,2025-10-27,2028-05-11,,,None,True
MAT100002,1020,V200012,2025-06-27,2028-02-16,,,None,True
MAT100003,1030,V200012,2025-07-27,2028-03-29,,,85,True
MAT100003,1030,V200008,2024-07-28,2028-05-21,,,15,True
MAT100003,1020,V200009,2026-01-21,2027-09-26,,,85,True
MAT100003,1020,V200013,2025-04-19,2028-05-20,,,7,True
MAT100003,1020,V200022,2025-06-01,2028-05-02,X,,7,False
MAT100004,1030,V200023,2024-11-23,2027-12-06,,,None,True


In [ ]:
dbutils.notebook.exit(f"02c_sourcing: SUCCESS — {cnt} rows.")

Notebook exited: 02c_sourcing: SUCCESS — 220 rows.
